In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from frechetdist import frdist
from keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
from scipy.spatial.distance import directed_hausdorff
from scipy.stats import gaussian_kde
import seaborn as sns
from shapely.geometry import Polygon
from sklearn.metrics import auc, roc_curve, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
import tensorflow as tf
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [3]:
import keras
print(keras.__version__)

2.2.4


In [4]:
import os
import sys

# Use this to load our modules.
module_path = os.path.abspath(os.path.join(os.getcwd(), '../../exGAN'))
if module_path not in sys.path:
    sys.path.append(module_path)

from model import LSTM_Traj_exGAN
from exGAN_module import exGAN

In [5]:
SEED = 2023
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.compat.v1.set_random_seed(SEED)

# Utils functions

In [6]:
def data_conversion(df, tid_col,test=False):
    """Converts input panda dataframe to one-hot-encoded Numpy array (locations are still in float)."""
    
    x = [[] for i in ['lat_lon', 'day', 'hour', 'category', 'mask','tid','count']]
    for tid in df[tid_col].unique():
        traj = df.loc[df[tid_col].isin([tid])]
        features = np.transpose(traj.loc[:, ['lat', 'lon', 'day', 'hour', 'category']].values)
        loc_list = []
        for i in range(0, len(traj)):
            lat = traj['lat'].values[i]
            lon = traj['lon'].values[i]
            loc_list.append(np.array([lat, lon], dtype=np.float64))
        x[0].append(loc_list)
        x[1].append(np.eye(7)[features[2].astype(np.int32)])
        x[2].append(np.eye(24)[features[3].astype(np.int32)])
        x[3].append(np.eye(10)[features[4].astype(np.int32)])
        x[4].append(np.ones(shape=(features[0].shape[0],1))) # mask
        
        if test:
            x[5].append(tid)
            x[6].append(traj.tid.count())
        

    converted_data = np.array([np.array(f, dtype=object) for f in x])
    return converted_data

def data_deconversion(x_test, x_prediction, df_test, max_length):
    """
    Converts one-hot-encoded Numpy array prediction (locations are still in float) to pd dataframe.

    x_test: original test set
    x_prediction: analog synthetic set
    
    """
    traj_attr_concat_list = []
    for attributes in x_prediction:
        traj_attr_list = []
        idx = 0
        for row in attributes:
            if row.shape == (max_length, 2): #If we have lat , lon
                traj_attr_list.append(row[max_length-x_test[6][idx][0]:])
            else:
                traj_attr_list.append(np.argmax(row[max_length-x_test[6][idx][0]:],axis=1).reshape(x_test[6][idx][0],1))
            idx += 1
        traj_attr_concat = np.concatenate(traj_attr_list)
        traj_attr_concat_list.append(traj_attr_concat)
    traj_data = np.concatenate(traj_attr_concat_list,axis=1)

    label = np.array(df_test['label']).reshape(-1,1)
    tid = np.array(df_test['tid']).reshape(-1,1)
    traj_data = np.concatenate([label,tid,traj_data],axis=1)
    df_traj_fin = pd.DataFrame(traj_data)
        
    df_traj_fin.columns = ['label','tid','lat','lon','day', 'hour', 'category','mask', 'cond']    
        
    del df_traj_fin['mask']
        
    df_traj_fin['tid'] = df_traj_fin['tid'].astype(np.int32)
    df_traj_fin['day'] = df_traj_fin['day'].astype(np.int32)
    df_traj_fin['hour'] = df_traj_fin['hour'].astype(np.int32)
    df_traj_fin['category'] = df_traj_fin['category'].astype(np.int32)
    df_traj_fin['label'] = df_traj_fin['label'].astype(np.int32)
    df_traj_fin['cond'] = df_traj_fin['cond'].astype(np.int32)

    return df_traj_fin

def scale(df_origin, GAN):
    df = df_origin.copy()
    lat_centroid = GAN.lat_centroid
    lon_centroid = GAN.lon_centroid

    scale_factor = GAN.scale_factor

    df["lat"] = df["lat"].apply(lambda x : x - lat_centroid)
    df["lon"] = df["lon"].apply(lambda x : x - lon_centroid)
    return df

def unscale(df, GAN):
    df_unscaled = df.copy()
    df_unscaled['lat'] = df_unscaled['lat'] + GAN.lat_centroid
    df_unscaled['lon'] = df_unscaled['lon'] + GAN.lon_centroid
    return df_unscaled

def Dist_1_1(test: pd.DataFrame, syn: pd.DataFrame, distance: str):
    syn_dist = []

    assert test["label"].unique().tolist() == syn["label"].unique().tolist() , "Users in test and syn aren't the same"

    for label in test["label"].unique():

        tidlist = list(test[test["label"] == label ]['tid'].unique())
        for  tid in  tidlist:

            real_traj = np.concatenate([np.array(test.loc[test['tid'] == tid]["lat"]).reshape(-1,1),
                                        np.array(test.loc[test['tid'] == tid]["lon"]).reshape(-1,1)],
                                    axis=1)
            syn_traj = np.concatenate([np.array(syn.loc[syn['tid'] == tid]["lat"]).reshape(-1,1),
                                    np.array(syn.loc[syn['tid'] == tid]["lon"]).reshape(-1,1)],
                                    axis=1)

            if distance == 'Haussdorff':
                syn_dist.append(directed_hausdorff(real_traj,syn_traj)[0])
            elif distance == 'Frechet':
                syn_dist.append(frdist(real_traj,syn_traj))

    return syn_Haus


def hours_1_1(test: pd.DataFrame, syn: pd.DataFrame):
    dis_cos = []

    assert test["label"].unique().tolist() == syn["label"].unique().tolist() , "Users in test and syn aren't the same"

    for label in test["label"].unique():

        tidlist = list(test[test["label"] == label ]['tid'].unique())
        
        for  tid in  tidlist:

            real_hour = np.array(test.loc[test['tid'] == tid]["hour"]).reshape(-1,1)
            syn_hour = np.array(syn.loc[syn['tid'] == tid]["hour"]).reshape(-1,1)


            dis_cos.append(np.linalg.norm(real_hour - syn_hour))
        
    return dis_cos 

def cat_1_1(test: pd.DataFrame, syn: pd.DataFrame):
    dis_cos = []

    assert test["label"].unique().tolist() == syn["label"].unique().tolist() , "Users in test and syn aren't the same"

    for label in test["label"].unique():

        tidlist = list(test[test["label"] == label ]['tid'].unique())
        
        for  tid in  tidlist:

            real_day = np.array(test.loc[test['tid'] == tid]["day"]).reshape(-1,1)
            syn_day = np.array(syn.loc[syn['tid'] == tid]["day"]).reshape(-1,1)
            
            real_cat = np.array(test.loc[test['tid'] == tid]["category"]).reshape(-1,1)
            syn_cat = np.array(syn.loc[syn['tid'] == tid]["category"]).reshape(-1,1)


            dis_cos.append(np.linalg.norm(real_day - syn_day) + np.linalg.norm(real_cat - syn_cat))
        
    return dis_cos 

def find_closest_cat(df_syn, trajectory):
    # trajectory does not contain any tid and label column
    dis_euclide = []
    tid_min = None
    mini = 99
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_day = np.array(df_syn.loc[df_syn['tid'] == tid]["day"]).reshape(-1,1)
            syn_category = np.array(df_syn.loc[df_syn['tid'] == tid]["category"]).reshape(-1,1)

            if len(syn_day) == len(np.array(trajectory["day"]).reshape(-1,1)) and len(syn_category) == len(np.array(trajectory["category"]).reshape(-1,1)) :
                tamp = (np.linalg.norm(np.array(trajectory["day"]).reshape(-1,1) - syn_day) + np.linalg.norm(np.array(trajectory["category"]).reshape(-1,1) - syn_category))
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

def find_closest_traj(df_syn, trajectory):
    # trajectory does not contain any tid and label column
    dis_euclide = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)

            if len(syn_lat) == len(tamp_lat) :
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                tamp = frdist(real_traj, syn_traj)
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

def matching(df_syn, df_real):
    count = 0
    for tid in tqdm(df_real['tid'].unique()):
        trajectory = df_real.loc[df_real['tid'] == tid]
        tid_min,_ = find_closest(df_syn, trajectory)
        if tid == tid_min:
            count += 1
    return count / df_real['tid'].nunique()

def synthesize(df, GAN, cond):
    exGAN_utils = exGAN()

    df_scaled = scale(df, GAN)

    x_test = data_conversion(df_scaled, "tid", test=True)
    x_test = [x_test[0],x_test[1],x_test[2],x_test[3],x_test[4],x_test[5].reshape(-1,1),x_test[6].reshape(-1,1)]
    X_test = [pad_sequences(f, GAN.max_length, padding='pre', dtype='float64') for f in x_test[:5]]

    #exGAN module
    category_index = exGAN_utils.average_value_by_category(X_test, mask=X_test[4])
    new_latlon, new_day, new_hour = exGAN_utils.conditional_label(X_test, X_test[3], category_index, X_test[4], cond=cond)
    for i, var in enumerate([new_latlon, new_day, new_hour]):
        X_test[i] = pad_sequences(var, GAN.max_length, padding='pre', dtype='float64')

    except_label_cond = np.zeros((X_test[0].shape[0], GAN.vocab_size['category']))
    except_label_cond[:, cond] = 1
    except_label_seq = np.repeat(except_label_cond[:, np.newaxis, :], GAN.max_length, axis=1) 
    X_test.append(except_label_seq.astype(np.float32))
        
    num_traj = X_test[0].shape[0]


    noise = np.random.normal(0, 1, (num_traj, 100))
    X_test.append(noise)

    prediction = GAN.generator.predict(X_test)
    
    # Reverse One-Hot encoding
    df_syn = data_deconversion(x_test, prediction, df_scaled, GAN.max_length)

    df_syn = unscale(df_syn, GAN)
    return df_syn

def find_closest_custom(df_syn, trajectory):
    distances = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)
            syn_cat = np.array(df_syn.loc[df_syn['tid'] == tid]["category"]).reshape(1,-1)
            syn_hour = np.array(df_syn.loc[df_syn['tid'] == tid]["hour"]).reshape(1,-1)
            syn_day = np.array(df_syn.loc[df_syn['tid'] == tid]["day"]).reshape(1,-1)

            if len(syn_lat) == len(tamp_lat) :
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                frech_dist = frdist(real_traj, syn_traj)
                cos_dist_cat = 1 - cosine_similarity(np.array(trajectory["category"]).reshape(1,-1), syn_cat)[0][0]
                cos_dist_hour = 1 - cosine_similarity(np.array(trajectory["hour"]).reshape(1,-1), syn_hour)[0][0]
                cos_dist_day = 1 - cosine_similarity(np.array(trajectory["day"]).reshape(1,-1), syn_day)[0][0]
                tamp = cos_dist_cat + cos_dist_hour + cos_dist_day + frech_dist
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

In [7]:
# global variables
PATH_TO_DATA_TRAIN = #Your data train path
PATH_TO_DATA_TEST = #Your data test path 
Frechet = False
Custom = True
cond=1

# Paper results

## 1. Standard MIA

### a. Split the dataset (trajectory level)

In [8]:
print(f'------------------------------- SPLIT in the GLOBAL dataset ---------------------------------------------------------------')
train_dataset = pd.read_csv(PATH_TO_DATA_TRAIN)
test_dataset = pd.read_csv(PATH_TO_DATA_TEST)
full_dataset = pd.concat([train_dataset_them,test_dataset_them], axis=0)
print(f"num users in the full dataset: {full_dataset['label'].nunique()} and total num of trajectories: {full_dataset['tid'].nunique()}")

split_cat = 'tid'
ids = full_dataset[split_cat].unique()
np.random.shuffle(ids)
split = int(len(ids)*0.5)
df_target, df_aux = full_dataset[full_dataset[split_cat].isin(ids[:split])], full_dataset[full_dataset[split_cat].isin(ids[split:])]

print(f"num users in the dataset target: {df_target['label'].nunique()} and total num of trajectories in it: {df_target['tid'].nunique()}")
print(f"num users in the dataset auxiliary: {df_aux['label'].nunique()} and total num of trajectories in it: {df_aux['tid'].nunique()}")

print(f'------------------------------- SPLITS in the TARGET dataset ---------------------------------------------------------------')

ids = df_target[split_cat].unique()
np.random.shuffle(ids)
split1 = int(len(ids)*0.7) 
split2 = int(len(ids)*0.90) 
df_target_train, Q_target_gen, Q_target_test = df_target[df_target[split_cat].isin(ids[:split1])], df_target[df_target[split_cat].isin(ids[split1:split2])],df_target[df_target[split_cat].isin(ids[split2:])]

print(f"num users in D target Train: {df_target_train['label'].nunique()} and total num of trajectories in it: {df_target_train['tid'].nunique()}")
print(f"num users in Q target for synthetization: {Q_target_gen['label'].nunique()} and total num of trajectories in it: {Q_target_gen['tid'].nunique()}")
print(f"num users in Q target for tests: {Q_target_test['label'].nunique()} and total num of trajectories in it: {Q_target_test['tid'].nunique()}")

print(f'------------------------------- SPLITS in the AUXILIARY dataset ---------------------------------------------------------------')

ids = df_aux[split_cat].unique()
np.random.shuffle(ids)
split1 = int(len(ids)*0.7)
split2 = int(len(ids)*0.90) 
df_aux_train, Q_aux_gen, Q_aux_test = df_aux[df_aux[split_cat].isin(ids[:split1])], df_aux[df_aux[split_cat].isin(ids[split1:split2])],df_aux[df_aux[split_cat].isin(ids[split2:])]

print(f"num users in D aux Train: {df_aux_train['label'].nunique()} and total num of trajectories in it: {df_aux_train['tid'].nunique()}")
print(f"num users in Q aux for synthetization: {Q_aux_gen['label'].nunique()} and total num of trajectories in it: {Q_aux_gen['tid'].nunique()}")
print(f"num users in Q aux for tests: {Q_aux_test['label'].nunique()} and total num of trajectories in it: {Q_aux_test['tid'].nunique()}")

------------------------------- SPLIT in the GLOBAL dataset ---------------------------------------------------------------
num users in the full dataset: 193 and total num of trajectories: 3079
num users in the dataset target: 193 and total num of trajectories in it: 1539
num users in the dataset auxiliary: 193 and total num of trajectories in it: 1540
------------------------------- SPLITS in the TARGET dataset ---------------------------------------------------------------
num users in D target Train: 193 and total num of trajectories in it: 1077
num users in Q target for synthetization: 149 and total num of trajectories in it: 308
num users in Q target for tests: 104 and total num of trajectories in it: 154
------------------------------- SPLITS in the AUXILIARY dataset ---------------------------------------------------------------
num users in D aux Train: 192 and total num of trajectories in it: 1078
num users in Q aux for synthetization: 158 and total num of trajectories in it:

### b. Train / load the models

In [9]:
# Global variables set by LSTMTrajGAN
latent_dim = 100
max_length = 144
exGAN_PATH = # Your saving path

keys = ['lat_lon', 'day', 'hour', 'category', 'mask', 'except_label']
vocab_size = {"lat_lon":2,"day":7,"hour":24,"category":10,"mask":1, "except_label":10}

n_epochs = 2000
n_batch_size = 128
n_sample_interval = 100

lat_centroid_aux = df_aux_train['lat'].sum()/len(df_aux_train)
lon_centroid_aux = df_aux_train['lon'].sum()/len(df_aux_train)

scale_factor_aux = max(max(abs(df_aux_train['lat'].max() - lat_centroid_aux),
                     abs(df_aux_train['lat'].min() - lat_centroid_aux),
                    ),
                 max(abs(df_aux_train['lon'].max() - lon_centroid_aux),
                     abs(df_aux_train['lon'].min() - lon_centroid_aux),
                    ))

gan_aux = LSTM_Traj_exGAN(latent_dim, keys, vocab_size, max_length, lat_centroid_aux, lon_centroid_aux, scale_factor_aux, SEED, all_params_path = exGAN_PATH)
gan_aux_path = f'{exGAN_PATH}\training_params\G_model_' + str(n_epochs) + '_paper_auxiliary.h5'

df_aux_train_normalized = df_aux_train.copy()
df_aux_train_normalized["lat"] = df_aux_train_normalized["lat"].apply(lambda x : x - lat_centroid_aux)
df_aux_train_normalized["lon"] = df_aux_train_normalized["lon"].apply(lambda x : x - lon_centroid_aux)
gan_aux.train(train=df_aux_train_normalized, epochs=n_epochs, batch_size=n_batch_size, sample_interval=n_sample_interval, adding = 'paper_auxiliary')

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Use tf.cast instead.
[1/2000] D Loss: 32.311329 | G Loss: 1095.867065
[2/2000] D Loss: 30.660217 | G Loss: 1070.480103
[3/2000] D Loss: 29.017048 | G Loss: 1067.426636
[4/2000] D Loss: 27.334770 | G Loss: 1056.730225
[5/2000] D Loss: 25.704731 | G Loss: 1055.514648
[6/2000] D Loss: 24.104668 | G Loss: 1046.412964
[7/2000] D Loss: 22.563242 | G Loss: 1043.203979
[8/2000] D Loss: 21.074759 | G Loss: 1040.813110
[9/2000] D Loss: 19.685413 | G Loss: 1035.572632
[10/2000] D Loss: 18.325417 | G Loss: 1032.680908
[11/2000] D Loss: 17.078663 | G Loss: 1034.997314
[12/2000] D Loss: 15.857815 | G Loss: 1025.476074
[13/2000] D Loss: 14.711731 | G Loss: 1023.086304
[14/2000] D Loss: 13.629662 | G Loss: 1024.692627
[15/2000] D Loss: 12.560976 | G Loss: 1019.498596
[16/2000] D Loss: 11.598583 | G Loss: 1020.960205
[17/2000] D Loss: 10.723531 | G Loss: 1017.710754
[18/2000] D Loss: 9.813156 | G Loss: 10

In [10]:
# Global variables set by LSTMTrajGAN
latent_dim = 100
max_length = 144

keys = ['lat_lon', 'day', 'hour', 'category', 'mask', 'except_label']
vocab_size = {"lat_lon":2,"day":7,"hour":24,"category":10,"mask":1, "except_label":10}

n_epochs = 2000
n_batch_size = 128
n_sample_interval = 100

lat_centroid_target = df_target_train['lat'].sum()/len(df_target_train)
lon_centroid_target = df_target_train['lon'].sum()/len(df_target_train)

scale_factor_target = max(max(abs(df_target_train['lat'].max() - lat_centroid_target),
                     abs(df_target_train['lat'].min() - lat_centroid_target),
                    ),
                 max(abs(df_target_train['lon'].max() - lon_centroid_target),
                     abs(df_target_train['lon'].min() - lon_centroid_target),
                    ))

gan_target = LSTM_Traj_exGAN(latent_dim, keys, vocab_size, max_length, lat_centroid_target, lon_centroid_target, scale_factor_target, SEED, all_params_path = exGAN_PATH)
gan_target_path = f'{exGAN_PATH}\training_params\G_model_' + str(n_epochs) + '_paper_target.h5'

df_target_train_normalized = df_target_train.copy()
df_target_train_normalized["lat"] = df_target_train_normalized["lat"].apply(lambda x : x - lat_centroid_target)
df_target_train_normalized["lon"] = df_target_train_normalized["lon"].apply(lambda x : x - lon_centroid_target)
gan_target.train(train=df_target_train_normalized, epochs=n_epochs, batch_size=n_batch_size, sample_interval=n_sample_interval, adding = 'paper_target')

[1/2000] D Loss: 32.327965 | G Loss: 1096.905273
[2/2000] D Loss: 30.660151 | G Loss: 1068.510864
[3/2000] D Loss: 29.016396 | G Loss: 1064.188477
[4/2000] D Loss: 27.331387 | G Loss: 1057.984619
[5/2000] D Loss: 25.704628 | G Loss: 1056.593262
[6/2000] D Loss: 24.105499 | G Loss: 1046.760010
[7/2000] D Loss: 22.565233 | G Loss: 1041.994751
[8/2000] D Loss: 21.096592 | G Loss: 1039.223999
[9/2000] D Loss: 19.686201 | G Loss: 1034.390137
[10/2000] D Loss: 18.348614 | G Loss: 1034.831665
[11/2000] D Loss: 17.074558 | G Loss: 1030.854004
[12/2000] D Loss: 15.860825 | G Loss: 1025.984985
[13/2000] D Loss: 14.742711 | G Loss: 1029.207153
[14/2000] D Loss: 13.612163 | G Loss: 1024.291992
[15/2000] D Loss: 12.597813 | G Loss: 1021.592468
[16/2000] D Loss: 11.604298 | G Loss: 1022.618530
[17/2000] D Loss: 10.708698 | G Loss: 1012.791260
[18/2000] D Loss: 9.827990 | G Loss: 1016.351562
[19/2000] D Loss: 8.998782 | G Loss: 1016.098511
[20/2000] D Loss: 8.271673 | G Loss: 1010.601562
[21/2000] D 

### c. Synthesize the datasets

In [11]:
df_syn_aux = synthesize(Q_aux_gen, gan_aux, cond)
df_syn_target = synthesize(Q_target_gen, gan_target, cond)

### d. Compute the threshold

In [12]:
n = 50
tids_to_remove_IN = np.random.choice(Q_aux_gen['tid'].unique(), size = (Q_aux_gen['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_aux_test['tid'].unique(), size = (Q_aux_test['tid'].nunique() - n), replace=False)
IN_data = Q_aux_gen.copy()
OUT_data = Q_aux_test.copy()
IN_data = IN_data[~IN_data['tid'].isin(tids_to_remove_IN)]
OUT_data = OUT_data[~OUT_data['tid'].isin(tids_to_remove_OUT)]

In [13]:
IN_dist_aux = []
for tid in tqdm(IN_data['tid'].unique()):
    trajectory = IN_data.loc[IN_data['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux, trajectory) 
    else:
        print('Error: metric not selected')
    IN_dist_aux.append(mini)

OUT_dist_aux = []
for tid in tqdm(OUT_data['tid'].unique()) :
    trajectory = OUT_data.loc[OUT_data['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux, trajectory) 
    else:
        print('Error: metric not selected')
    OUT_dist_aux.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:37<00:00,  1.33it/s]


In [14]:
# KDE estimation
kde_train = gaussian_kde(IN_dist_aux)
kde_test = gaussian_kde(OUT_dist_aux)

x_grid = np.linspace(min(min(IN_dist_aux), min(OUT_dist_aux)), max(max(IN_dist_aux), max(OUT_dist_aux)), 1000)

mode_train = x_grid[np.argmax(kde_train(x_grid))]
mode_test = x_grid[np.argmax(kde_test(x_grid))]

threshold = (mode_test - mode_train)/2 + mode_train

### e. Finish the attack and compute the accuracy

In [15]:
n = 50
tids_to_remove_IN = np.random.choice(Q_target_gen['tid'].unique(), size = (Q_target_gen['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_target_test['tid'].unique(), size = (Q_target_test['tid'].nunique() - n), replace=False)
IN_data = Q_target_gen.copy()
OUT_data = Q_target_test.copy()
IN_data_target = IN_data[~IN_data['tid'].isin(tids_to_remove_IN)]
OUT_data_target = OUT_data[~OUT_data['tid'].isin(tids_to_remove_OUT)]

In [16]:
IN_dist_target = []
for tid in tqdm(IN_data_target['tid'].unique()):
    trajectory = IN_data_target.loc[IN_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory)
    else:
        print('Error: metric not selected')
    IN_dist_target.append(mini)

OUT_dist_target = []
for tid in tqdm(OUT_data_target['tid'].unique()) :
    trajectory = OUT_data_target.loc[OUT_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    OUT_dist_target.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:37<00:00,  1.34it/s]


In [17]:
true_labels = []
predictions = []
for dist in OUT_dist_target:
    true_labels.append(0)
    predictions.append(1 if dist <= threshold else 0)
for dist in IN_dist_target:
    true_labels.append(1)
    predictions.append(1 if dist <= threshold else 0)

In [18]:
accuracy_score(true_labels, predictions)

0.84

In [19]:
SEED

2023

### Results Main exGAN FC

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.70|
|2024|0.69|
|2025|0.78|
|2026|0.79|

Mean: 0.740 Std: 0.045

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.84|
|2024|0.93|
|2025|0.91|
|2026|0.52|

Mean: 0.800 Std: 0.165

### Results Main exGAN Geolife

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.83|
|2024|0.84|
|2025|0.83|
|2026|0.83|

Mean: 0.833 Std: 0.004

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.89|
|2024|0.92|
|2025|0.96|
|2026|0.89|

Mean: 0.915 Std: 0.029

## 2. Masked MIA

### a. Utils

In [20]:
def mask(df, ratio):
    df_masked = df.copy().reset_index(drop=True)
    for tid in tqdm(df['tid'].unique()):
        trajectory = df_masked.loc[df_masked['tid'] == tid]
        mask_indices_tid = trajectory.sample(frac=ratio).index
        df_masked.loc[mask_indices_tid, ['lat','lon']] = np.nan
    return df_masked

def find_closest_traj_masked(df_syn, trajectory):
    # trajectory does not contain any tid and label column
    dis_euclide = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    # we retrieve the mask of the trajectory, while keeping the indices of the mask
    mask = np.isnan(real_traj[:, 0]) & np.isnan(real_traj[:, 1])
    indices_mask = np.where(mask)[0]
    real_traj_masked = np.delete(real_traj, indices_mask, axis=0)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)
            
            if len(syn_lat) == len(tamp_lat) :
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                # we mask the synthetic trajectory the same way, to keep consistency between the traces 
                syn_traj_masked = np.delete(syn_traj, indices_mask, axis=0)
                tamp = frdist(real_traj_masked, syn_traj_masked)
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

def find_closest_custom_masked(df_syn, trajectory):
    distances = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    # we retrieve the mask of the trajectory, while keeping the indices of the mask
    mask = np.isnan(real_traj[:, 0]) & np.isnan(real_traj[:, 1])
    indices_mask = np.where(mask)[0]
    real_traj_masked = np.delete(real_traj, indices_mask, axis=0)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)
            syn_cat = np.array(df_syn.loc[df_syn['tid'] == tid]["category"]).reshape(1,-1)
            syn_hour = np.array(df_syn.loc[df_syn['tid'] == tid]["hour"]).reshape(1,-1)
            syn_day = np.array(df_syn.loc[df_syn['tid'] == tid]["day"]).reshape(1,-1)

            if len(syn_lat) == len(tamp_lat) :
                syn_cat_masked = np.delete(syn_cat, indices_mask, axis=1)
                syn_hour_masked = np.delete(syn_hour, indices_mask, axis=1)
                syn_day_masked = np.delete(syn_day, indices_mask, axis=1)
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                syn_traj_masked = np.delete(syn_traj, indices_mask, axis=0)
                frech_dist = frdist(real_traj_masked, syn_traj_masked)
                real_cat_masked = np.delete(np.array(trajectory["category"]).reshape(1,-1), indices_mask, axis=1)
                real_hour_masked = np.delete(np.array(trajectory["hour"]).reshape(1,-1), indices_mask, axis=1)
                real_day_masked = np.delete(np.array(trajectory["day"]).reshape(1,-1), indices_mask, axis=1)
                cos_dist_cat = 1 - cosine_similarity(real_cat_masked, syn_cat_masked)[0][0]
                cos_dist_hour = 1 - cosine_similarity(real_hour_masked, syn_hour_masked)[0][0]
                cos_dist_day = 1 - cosine_similarity(real_day_masked, syn_day_masked)[0][0]
                tamp = cos_dist_cat + cos_dist_hour + cos_dist_day + frech_dist
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

### b. Mask application on the target data

In [21]:
# First, consider the parameter alpha to describe the purcentage of the trace we leave out
alpha = 0.75
# Second, remove the information from the targets
OUT_data_masked = mask(OUT_data_target, alpha)
IN_data_masked = mask(IN_data_target, alpha)

100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 671.71it/s]


In [22]:
OUT_dist_mask = []
for tid in tqdm(OUT_data_masked['tid'].unique()) :
    trajectory = OUT_data_masked.loc[OUT_data_masked['tid'] == tid]
    if Frechet:
        tid_min, mini = find_closest_traj_masked(df_syn_target, trajectory) 
    elif Custom:
        tid_min, mini = find_closest_custom_masked(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    OUT_dist_mask.append(mini)
IN_dist_mask = []
for tid in tqdm(IN_data_masked['tid'].unique()):
    trajectory = IN_data_masked.loc[IN_data_masked['tid'] == tid]
    if Frechet:
        tid_min, mini = find_closest_traj_masked(df_syn_target, trajectory)
    elif Custom:
        tid_min, mini = find_closest_custom_masked(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    IN_dist_mask.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:36<00:00,  1.38it/s]


In [23]:
true_labels = []
predictions_masked = []
for dist in OUT_dist_mask:
    true_labels.append(0)
    predictions_masked.append(1 if dist <= threshold else 0)
for dist in IN_dist_mask:
    true_labels.append(1)
    predictions_masked.append(1 if dist <= threshold else 0)

In [24]:
accuracy_score(true_labels, predictions_masked)

0.73

In [25]:
SEED

2023

### Results Mask exGAN FC

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.73|
|2024|0.73|
|2025|0.75|
|2026|0.75|

Mean: 0.740 Std: 0.010

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.73|
|2024|0.70|
|2025|0.69|
|2026|0.52|

Mean: 0.660 Std: 0.082

### Results Mask exGAN Geolife

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.79|
|2024|0.78|
|2025|0.81|
|2026|0.85|

Mean: 0.808 Std: 0.027

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.73|
|2024|0.76|
|2025|0.78|
|2026|0.79|

Mean: 0.765 Std: 0.023

## 3. Synthetic only MIA

In this specific setting, we now don't let access to the auxiliary information. 
Instead, we provide a greater synthetic dataset (by adding D_aux_train, Q_aux_gen and Q_aux_test to Q_target_gen).

To facilitate the comprehension of the attack, we keep the split, and we generate the synthetic information for each of them. 
The goal of the attacker will remain to infer the mmebership in Q_target_gen, but with the use of D_aux_train_synth, Q_aux_gen_synth and Q_aux_test_synth

### a. Production of the synthetic information

In [26]:
df_syn_target = synthesize(Q_target_gen, gan_target, cond)
df_aux_train_syn = synthesize(df_aux_train, gan_target, cond)
Q_aux_gen_syn = synthesize(Q_aux_gen, gan_target, cond)
Q_aux_test_syn = synthesize(Q_aux_test, gan_target, cond)

### b. Recomputation of the new exGAN based only on the synthetic information

In [27]:
# Global variables set by LSTMTrajGAN
latent_dim = 100
max_length = 144

keys = ['lat_lon', 'day', 'hour', 'category', 'mask', 'except_label']
vocab_size = {"lat_lon":2,"day":7,"hour":24,"category":10,"mask":1, "except_label":10}

n_epochs = 2000
n_batch_size = 128
n_sample_interval = 100

lat_centroid_aux_syn = df_aux_train_syn['lat'].sum()/len(df_aux_train_syn)
lon_centroid_aux_syn = df_aux_train_syn['lon'].sum()/len(df_aux_train_syn)

scale_factor_aux_syn = max(max(abs(df_aux_train_syn['lat'].max() - lat_centroid_aux_syn),
                               abs(df_aux_train_syn['lat'].min() - lat_centroid_aux_syn),
                    ),
                           max(abs(df_aux_train_syn['lon'].max() - lon_centroid_aux_syn),
                               abs(df_aux_train_syn['lon'].min() - lon_centroid_aux_syn),
                            ))

gan_aux_syn = LSTM_Traj_exGAN(latent_dim, keys, vocab_size, max_length, lat_centroid_aux_syn, lon_centroid_aux_syn, scale_factor_aux_syn, SEED, all_params_path = exGAN_PATH)
gan_aux_syn_path = f'{exGAN_PATH}\training_params\G_model_' + str(n_epochs) + '_paper_auxiliary_synthetic_only.h5'

df_aux_train_syn_normalized = df_aux_train_syn.copy()
df_aux_train_syn_normalized["lat"] = df_aux_train_syn_normalized["lat"].apply(lambda x : x - lat_centroid_aux_syn)
df_aux_train_syn_normalized["lon"] = df_aux_train_syn_normalized["lon"].apply(lambda x : x - lon_centroid_aux_syn)
gan_aux_syn.train(train=df_aux_train_syn_normalized, epochs=n_epochs, batch_size=n_batch_size, sample_interval=n_sample_interval, adding = 'paper_auxiliary_synthetic_only')

[1/2000] D Loss: 32.306347 | G Loss: 1082.809326
[2/2000] D Loss: 30.662491 | G Loss: 1065.369629
[3/2000] D Loss: 29.028194 | G Loss: 1060.246948
[4/2000] D Loss: 27.343037 | G Loss: 1050.119751
[5/2000] D Loss: 25.710882 | G Loss: 1048.563721
[6/2000] D Loss: 24.106264 | G Loss: 1039.713501
[7/2000] D Loss: 22.566265 | G Loss: 1035.167480
[8/2000] D Loss: 21.077620 | G Loss: 1033.549194
[9/2000] D Loss: 19.690483 | G Loss: 1024.231201
[10/2000] D Loss: 18.322390 | G Loss: 1022.651733
[11/2000] D Loss: 17.078846 | G Loss: 1015.514343
[12/2000] D Loss: 15.862366 | G Loss: 999.060242
[13/2000] D Loss: 14.717302 | G Loss: 1000.483582
[14/2000] D Loss: 13.621954 | G Loss: 990.588806
[15/2000] D Loss: 12.543489 | G Loss: 978.911987
[16/2000] D Loss: 11.593880 | G Loss: 964.598389
[17/2000] D Loss: 10.688259 | G Loss: 959.920227
[18/2000] D Loss: 9.795084 | G Loss: 940.394043
[19/2000] D Loss: 8.991962 | G Loss: 949.835388
[20/2000] D Loss: 8.202055 | G Loss: 955.370789
[21/2000] D Loss: 7.

### c. Synthesize attacker datasets

In [28]:
df_syn_aux_synthetic_only = synthesize(Q_aux_gen_syn, gan_aux_syn, cond)

### d. Compute the threshold using the synthetic information

In [29]:
n = 50
tids_to_remove_IN = np.random.choice(Q_aux_gen_syn['tid'].unique(), size = (Q_aux_gen_syn['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_aux_test_syn['tid'].unique(), size = (Q_aux_test_syn['tid'].nunique() - n), replace=False)
IN_data_syn = Q_aux_gen_syn.copy()
OUT_data_syn = Q_aux_test_syn.copy()
IN_data_syn = IN_data_syn[~IN_data_syn['tid'].isin(tids_to_remove_IN)]
OUT_data_syn = OUT_data_syn[~OUT_data_syn['tid'].isin(tids_to_remove_OUT)]

In [30]:
IN_dist_aux_syn = []
for tid in tqdm(IN_data_syn['tid'].unique()):
    trajectory = IN_data_syn.loc[IN_data_syn['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux_synthetic_only, trajectory)
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux_synthetic_only, trajectory)
    else:
        print('Error: metric not selected')
    IN_dist_aux_syn.append(mini)

OUT_dist_aux_syn = []
for tid in tqdm(OUT_data_syn['tid'].unique()) :
    trajectory = OUT_data_syn.loc[OUT_data_syn['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux_synthetic_only, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux_synthetic_only, trajectory) 
    else:
        print('Error: metric not selected')
    OUT_dist_aux_syn.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:37<00:00,  1.34it/s]


In [31]:
# KDE estimation
kde_train = gaussian_kde(IN_dist_aux_syn)
kde_test = gaussian_kde(OUT_dist_aux_syn)

x_grid = np.linspace(min(min(IN_dist_aux_syn), min(OUT_dist_aux_syn)), max(max(IN_dist_aux_syn), max(OUT_dist_aux_syn)), 1000)

mode_train = x_grid[np.argmax(kde_train(x_grid))]
mode_test = x_grid[np.argmax(kde_test(x_grid))]

threshold = mode_test - abs(mode_train - mode_test)/2

### e. Finish the synthetic only attack and compute the accuracy

In [32]:
n = 50
tids_to_remove_IN = np.random.choice(Q_target_gen['tid'].unique(), size = (Q_target_gen['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_target_test['tid'].unique(), size = (Q_target_test['tid'].nunique() - n), replace=False)
IN_data = Q_target_gen.copy()
OUT_data = Q_target_test.copy()
IN_data_target = IN_data[~IN_data['tid'].isin(tids_to_remove_IN)]
OUT_data_target = OUT_data[~OUT_data['tid'].isin(tids_to_remove_OUT)]

In [33]:
IN_dist_target = []
for tid in tqdm(IN_data_target['tid'].unique()):
    trajectory = IN_data_target.loc[IN_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    IN_dist_target.append(mini)

OUT_dist_target = []
for tid in tqdm(OUT_data_target['tid'].unique()) :
    trajectory = OUT_data_target.loc[OUT_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    OUT_dist_target.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:37<00:00,  1.34it/s]


In [34]:
true_labels = []
predictions = []
for dist in OUT_dist_target:
    true_labels.append(0)
    predictions.append(1 if dist <= threshold else 0)
for dist in IN_dist_target:
    true_labels.append(1)
    predictions.append(1 if dist <= threshold else 0)

In [35]:
accuracy_score(true_labels, predictions)

0.53

In [36]:
SEED

2023

### Results Synthetic Only exGAN FC

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.65|
|2024|0.77|
|2025|0.69|
|2026|0.71|

Mean: 0.705 Std: 0.043

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.53|
|2024|0.59|
|2025|0.83|
|2026|0.85|

Mean: 0.700 Std: 0.142

### Results Synthetic Only exGAN Geolife

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.82|
|2024|0.71|
|2025|0.81|
|2026|0.76|

Mean: 0.775 Std: 0.044

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.96|
|2024|0.94|
|2025|0.94|
|2026|0.89|

Mean: 0.933 Std: 0.026